In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# สร้าง Spark Session
spark = SparkSession.builder \
    .appName("Read Parquet") \
    .getOrCreate()

# Post

In [ ]:
# Read Title
col_post_selected = ['content_categories',
                     'created_utc',
                     'selftext',
                     'subreddit',
                     'title',
                     'upvote_ratio',
                     'ups',
                     'downs',
                     'view_count']

paths = ["gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_Anthropic/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_Bard/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ChatGPT/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ChatGPTPro/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_ClaudeAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_DeepSeek/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_GeminiAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_OpenAI/part*",
         "gs://reddit-ai-2/flattened_data/all_posts/subreddit=r_grok/part*"]


post = spark.read.parquet(*paths).select(col_post_selected)

In [ ]:
post.printSchema()

In [ ]:
# Count Null for Each Columns
from pyspark.sql.functions import isnan, isnull, when, count

null_count = post.select([isnull(c).alias(c) for c in post.columns])

null_count.show()

In [ ]:
# Drop row when title is missing
post = post.dropna(how='any', subset = 'title')

post.count()

In [ ]:
# Sampling 10,000 Rows for Numerics EDA
post_sampled = post.sample(withReplacement=False, fraction=(10000/post.count()), seed=42)

import pandas as pd

post_sampled = post_sampled.toPandas()

In [ ]:
from google.cloud import storage

# 1. ตั้งค่าพื้นฐาน Bucket Name = 'A'
target_path = "gs://reddit-ai-2/process_data/Sampling_Data_for_EDA/Sampling_Post_for_EDA.csv"

# เขียนไฟล์ลง GCS โดยตรง
post_sampled.to_csv(target_path, index=False)

print(f"บันทึกไฟล์ไปที่ {target_path} เรียบร้อยแล้ว")

# Comment

In [ ]:
# Read Title
col_post_selected = ['body',
                     'created_utc',
                     'is_submitter',
                     'removal_reason',
                     'subreddit',
                     'score',
                     'ups',
                     ]

paths = ["gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_Anthropic/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_Bard/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ChatGPT/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ChatGPTPro/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_ClaudeAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_DeepSeek/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_GeminiAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_OpenAI/part*",
         "gs://reddit-ai-2/flattened_data/all_comments/subreddit=r_grok/part*"]


comment = spark.read.parquet(*paths).select(col_post_selected)

In [ ]:
comment.printSchema()

In [ ]:
# Count Null for Each Columns
from pyspark.sql.functions import isnan, isnull, when, count

null_count = comment.select([isnull(c).alias(c) for c in comment.columns])

null_count.show()

In [ ]:
# Drop row when title is missing
comment = comment.dropna(how='any', subset = 'body')

comment.count()

In [ ]:
# Sampling 10,000 Rows for Numerics EDA
comment_sampled = comment.sample(withReplacement=False, fraction=(10000/comment.count()), seed=42)

import pandas as pd

comment_sampled = comment_sampled.toPandas()

In [ ]:
from google.cloud import storage

# 1. ตั้งค่าพื้นฐาน Bucket Name = 'A'
target_path = "gs://reddit-ai-2/process_data/Sampling_Data_for_EDA/Sampling_Comment_for_EDA.csv"

# เขียนไฟล์ลง GCS โดยตรง
comment_sampled.to_csv(target_path, index=False)

print(f"บันทึกไฟล์ไปที่ {target_path} เรียบร้อยแล้ว")

In [ ]:
spark.stop()